# Behavior Prior (RGM) Rollout Visualization

Load a `distill_prior` checkpoint, roll out the RGM from specified clip/frame
start positions at multiple noise levels, and render each rollout as video.

In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import jax
import jax.numpy as jnp
import numpy as np
from jax import random
from pathlib import Path

import imageio
from IPython.display import Video, display

## Configuration

In [2]:
# ============ EDIT THESE ============

# Path to distill_prior checkpoint
CHECKPOINT_PATH = "/home/mila/a/aidan.sirbu/SCAMPER/scratch/distill_prior_checkpoints/260406_130524_381481"
CHECKPOINT_STEP = 54  # None = latest

# Number of rollouts per (noise_level, start_position) combination
NUM_ROLLOUTS = 5

# Noise levels: list of fixed log-variance values.
# Lower = less noise (more deterministic), higher = more stochastic.
# std = exp(0.5 * logvar), so logvar=-4 -> std~0.14, logvar=-2 -> std~0.37, logvar=0 -> std=1.0
# Use None for network-predicted logvar, "deterministic" for mean-only (no sampling).
NOISE_LEVELS = ["deterministic", -4.0, -3.0, -2.0, -1.0, 0.0, None]

# Starting positions as (clip_idx, start_frame) tuples
CLIP_FRAME_STARTS = [
    (0, 0),
    (1, 0),
    (2, 0),
    (3, 0),
    (4, 0),
    (5, 0),
]

# Rollout settings
MAX_STEPS = 1000
SEED = 42

# Rendering settings
RENDER_FPS = 50
CAMERA_NAME = "close_profile-rodent"
RENDER_HEIGHT = 720
RENDER_WIDTH = 1280

# Output directory for saved videos
OUTPUT_DIR = Path("/home/mila/a/aidan.sirbu/SCAMPER/notebooks/plots/behavior_prior_rollouts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load checkpoint and create environment

In [3]:
from scamper.scripts.collect_rollout_dataset_distill import (
    load_distill_prior_checkpoint_for_rollouts,
    create_rgm_policy_fn,
)
from scamper.scripts.collect_rollout_dataset import (
    fix_config_paths,
    create_environment,
    get_observation_sizes,
)

# Load checkpoint
cfg, policy_params = load_distill_prior_checkpoint_for_rollouts(
    CHECKPOINT_PATH, CHECKPOINT_STEP
)

# Fix paths and create environment
cfg = fix_config_paths(cfg)
env, reference_clips = create_environment(cfg)

reference_obs_size, proprioceptive_obs_size, total_obs_size = get_observation_sizes(cfg)

print(f"Action size: {cfg.network_config.action_size}")
print(f"Intention size: {cfg.network_config.intention_size}")
print(f"Proprioceptive obs size: {proprioceptive_obs_size}")
print(f"Reference obs size: {reference_obs_size}")
print(f"Number of reference clips: {reference_clips.qpos.shape[0]}")

Loading distill_prior checkpoint from /home/mila/a/aidan.sirbu/SCAMPER/scratch/distill_prior_checkpoints/260406_130524_381481 at step 54
Action size: 38
Intention size: 16
Proprioceptive obs size: 277
Reference obs size: 640
Number of reference clips: 842


## Rollout helpers

In [ ]:
from mujoco import mjx
from scamper.agent.observation_utils import flatten_obs_dict


def create_clip_state(env, clip_idx, start_frame, rng_key):
    """Create environment state initialized from a specific clip and frame."""
    state = env.reset(rng_key)
    unwrapped_env = env
    while hasattr(unwrapped_env, "_env"):
        unwrapped_env = unwrapped_env._env

    reference = unwrapped_env.reference_clips.at(clip=clip_idx, frame=start_frame)
    mjx_model = env.mjx_model
    data = state.data.replace(qpos=reference.qpos)
    data = data.replace(qvel=jnp.zeros(mjx_model.nv))
    data = mjx.forward(mjx_model, data)

    info = dict(state.info)
    info["start_frame"] = start_frame
    info["reference_clip"] = clip_idx
    state = state.replace(data=data, info=info)
    return state


def run_rollouts(env, policy_fn, starting_state, max_steps, rng_keys):
    """Run vmapped rollouts from a starting state and return all states."""
    jit_step = jax.jit(env.step)

    def single_rollout_fn(rng_key):
        state = starting_state

        def step_fn(carry, _):
            state, key = carry
            key, key_action = random.split(key)

            if hasattr(state.obs, "get") or isinstance(state.obs, dict):
                flat_obs = flatten_obs_dict(state.obs)
                proprio = flat_obs["proprioception"]
            else:
                proprio = state.obs

            action, _ = policy_fn(proprio, key_action)
            next_state = jit_step(state, action)

            return (next_state, key), next_state

        initial_carry = (state, rng_key)
        _, all_states = jax.lax.scan(step_fn, initial_carry, None, length=max_steps)

        return all_states

    vmapped_rollout = jax.jit(jax.vmap(single_rollout_fn))
    all_states = vmapped_rollout(rng_keys)
    jax.tree_util.tree_map(lambda x: x.block_until_ready(), all_states)
    return all_states


def get_unwrapped_env(env):
    """Unwrap to the underlying Imitation environment."""
    while hasattr(env, "_env"):
        env = env._env
    return env


def render_rollout(env, states, num_steps, camera, height=480, width=640):
    """Render a single rollout to a list of frames."""
    states_list = []
    for i in range(num_steps):
        state_i = jax.tree_util.tree_map(lambda x: x[i], states)
        states_list.append(state_i)
    # Call render on the unwrapped Imitation env to access render_ghost kwarg
    unwrapped = get_unwrapped_env(env)
    return unwrapped.render(
        states_list, camera=camera, height=height, width=width, render_ghost=False
    )


def noise_level_name(noise_level):
    """Human-readable name for a noise level."""
    if noise_level == "deterministic":
        return "deterministic"
    elif noise_level is None:
        return "predicted_logvar"
    else:
        return f"logvar_{noise_level}"


print("Helpers ready.")

## Run rollouts and render

For each noise level and each clip/frame start position, run `NUM_ROLLOUTS`
rollouts using the RGM, render them to `.mp4`, and display inline.

In [ ]:
import time

rng = random.PRNGKey(SEED)
video_paths = []

for noise_level in NOISE_LEVELS:
    nl_name = noise_level_name(noise_level)

    # Build RGM policy for this noise level
    if noise_level == "deterministic":
        rgm_policy = create_rgm_policy_fn(
            policy_params, cfg, fixed_logvar=0.0, deterministic=True
        )
    else:
        rgm_policy = create_rgm_policy_fn(
            policy_params, cfg, fixed_logvar=noise_level, deterministic=False
        )

    for clip_idx, start_frame in CLIP_FRAME_STARTS:
        pose_name = f"clip{clip_idx}_frame{start_frame}"
        print(f"\n{'='*60}")
        print(f"Noise: {nl_name} | Start: {pose_name} | Rollouts: {NUM_ROLLOUTS}")
        print(f"{'='*60}")

        # Create starting state
        rng, rng_init, rng_rollout = random.split(rng, 3)
        starting_state = create_clip_state(env, clip_idx, start_frame, rng_init)

        # Number of rollouts (1 for deterministic since they're identical)
        n = 1 if noise_level == "deterministic" else NUM_ROLLOUTS
        rollout_keys = random.split(rng_rollout, n)

        # Run rollouts
        t0 = time.time()
        all_states = run_rollouts(
            env, rgm_policy, starting_state, MAX_STEPS, rollout_keys
        )
        elapsed = time.time() - t0
        print(f"  Time: {elapsed:.1f}s")

        # Render each rollout
        for ri in range(n):
            rollout_states = jax.tree_util.tree_map(lambda x: x[ri], all_states)

            frames = render_rollout(
                env, rollout_states, MAX_STEPS,
                camera=CAMERA_NAME, height=RENDER_HEIGHT, width=RENDER_WIDTH,
            )

            video_filename = f"{nl_name}_{pose_name}_rollout{ri:02d}.mp4"
            video_path = OUTPUT_DIR / video_filename
            with imageio.get_writer(str(video_path), fps=RENDER_FPS) as writer:
                for frame in frames:
                    writer.append_data(frame)

            video_paths.append(video_path)
            print(f"  Saved: {video_filename}")

        # Free GPU memory
        del all_states

print(f"\nDone! {len(video_paths)} videos saved to {OUTPUT_DIR}")

## Display videos

In [ ]:
for vp in video_paths:
    print(f"\n{vp.name}")
    display(Video(str(vp), embed=True, width=RENDER_WIDTH))